In [1]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

## 1. Load Geocoded Data

In [2]:
df = pd.read_csv('../data/processed/geocoded_mumbai_indore_property_data.csv')

print(f"Initial dataset shape: {df.shape}")
print(f"Columns:\n{list(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum()}")

display(df.head())

Initial dataset shape: (440, 10)
Columns:
['Location', 'Property_Type', 'Price_INR', 'Area_sqft', 'BHK', 'Bathrooms', 'City', 'price_per_sqft', 'Latitude', 'Longitude']

Missing values:
Location           0
Property_Type      0
Price_INR          0
Area_sqft          0
BHK                0
Bathrooms          0
City               0
price_per_sqft     0
Latitude          85
Longitude         85
dtype: int64


,Location,Property_Type,Price_INR,Area_sqft,BHK,Bathrooms,City,price_per_sqft,Latitude,Longitude
0,Lower Parel,Apartment,53100000.0,1479.0,2,2.0,Mumbai,35902.64,18.9953,72.8300
1,Lower Parel,Apartment,63100000.0,2616.0,3,3.0,Mumbai,24120.80,18.9953,72.8300
2,Bandra East,Apartment,6700000.0,3578.0,3,4.0,Mumbai,1872.55,19.0600,72.8544
3,Juhu,Apartment,87900000.0,2579.0,2,2.0,Mumbai,34082.98,19.1048,72.8267
4,Worli,Apartment,93600000.0,1162.0,1,2.0,Mumbai,80550.77,19.0069,72.8156


In [3]:
if 'price_per_sqft' in df.columns:
    df = df.drop('price_per_sqft', axis=1, errors='ignore')
    print("Dropped price_per_sqft to prevent data leakage")

Dropped price_per_sqft to prevent data leakage


## 2. Define Features and Handle Missing Values via Pipeline

In [4]:
target = 'Price_INR'

# Define final features
categorical_features = ['City', 'Location', 'Property_Type']
numerical_features = ['Area_sqft', 'BHK', 'Bathrooms', 'Latitude', 'Longitude']
print(f"Final Categorical Features: {categorical_features}")
print(f"Final Numerical Features: {numerical_features}")

Final Categorical Features: ['City', 'Location', 'Property_Type']
Final Numerical Features: ['Area_sqft', 'BHK', 'Bathrooms', 'Latitude', 'Longitude']


## 3. Handle Domain Outliers

In [5]:
initial_rows = len(df)

# Domain-based outlier/invalid value removal
df = df[(df['Price_INR'] > 0) & (df['Area_sqft'] > 0)]
df = df[(df['BHK'] > 0) & (df['Bathrooms'] > 0)]

rows_removed = initial_rows - len(df)
print(f"Rows before outlier handling: {initial_rows}")
print(f"Rows removed (domain checks): {rows_removed}")
print(f"Rows remaining: {len(df)}")

Rows before outlier handling: 440
Rows removed (domain checks): 0
Rows remaining: 440


## 4. Construct Scikit-Learn Pipeline and Save Preprocessor

In [6]:
# Define unified preprocessing steps
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, numerical_features),
        ('cat', categorical_pipeline, categorical_features)
    ],
    remainder='passthrough'
)

X_raw = df[categorical_features + numerical_features]

# We must ensure X_raw order is correctly fed: [categorical_features] + [numerical_features]
# The preprocessor output will be num_features_transformed followed by cat_features_transformed.
X_encoded = preprocessor.fit_transform(X_raw)

os.makedirs('../model', exist_ok=True)
with open('../model/preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)
print("Saved preprocessor to: model/preprocessor.pkl")

# Create the explicit ML-ready dataset CSV for direct model training/inspection
try:
    encoded_columns = preprocessor.get_feature_names_out()
except AttributeError:
    encoded_columns = [f"f_{i}" for i in range(X_encoded.shape[1])]

df_encoded = pd.DataFrame(X_encoded, columns=encoded_columns)
df_encoded[target] = df[target].values

os.makedirs('../data/processed', exist_ok=True)
output_path = '../data/processed/ml_ready_mumbai_indore_data.csv'
df_encoded.to_csv(output_path, index=False)
print(f"Saved ml-ready dataset to: {output_path}")

Saved preprocessor to: model/preprocessor.pkl
Saved ml-ready dataset to: ../data/processed/ml_ready_mumbai_indore_data.csv


## 5. Train-Test Split

In [7]:
X = df_encoded.drop(columns=[target])
y = df_encoded[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (352, 30)
X_test shape: (88, 30)
y_train shape: (352,)
y_test shape: (88,)


In [8]:
print("--- Final Validation ---")
print(f"Final feature count: {X.shape[1]}")
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")
print(f"Missing values in X_train: {X_train.isnull().sum().sum()}")

--- Final Validation ---
Final feature count: 30
Training samples: 352
Testing samples: 88
Missing values in X_train: 0
